In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import pickle
import torch
import esm
from tqdm import tqdm
import time
import pickle
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

### Load Info About Wildtype Kinases

In [10]:
kinase_sequences = pd.read_csv("wildtype_kinase_data.csv")
kinase_sequences = kinase_sequences.rename(columns = {"Unnamed: 0" : "Protein"})

In [11]:
kinase_sequences.head()

,Protein,From,Entry,Reviewed,Entry Name,Protein names,Gene Names,Organism,Length,Domain [FT],Sequence,PubMed ID,KD start,KD end,KD2 start,KD2 end,KD Sequence,KD2 Sequence
0,ABL1,ABL1_HUMAN,P00519,reviewed,ABL1_HUMAN,Tyrosine-protein kinase ABL1 (EC 2.7.10.2) (Ab...,ABL1 ABL JTK7,Homo sapiens (Human),1130,"DOMAIN 61..121; /note=""SH3""; /evidence=""ECO:00...",MLEICLKLVGCKSKKGLSSSSSCYLEEALQRPVASDFEPQGLSEAA...,3021337; 2687768; 7665185; 15164053; 15489334;...,242,493,NaN,NaN,ITMKHKLGGGQYGEVYEGVWKKYSLTVAVKTLKEDTMEVEEFLKEA...,#NUM!
1,AKT1,AKT1_HUMAN,P31749,reviewed,AKT1_HUMAN,RAC-alpha serine/threonine-protein kinase (EC ...,AKT1 PKB RAC,Homo sapiens (Human),480,"DOMAIN 5..108; /note=""PH""; /evidence=""ECO:0000...",MSDVAIVKEGWLHKRGEYIKTWRPRYFLLKNDGTFIGYKERPQDVD...,1851997; 11508278; 14702039; 19054851; 1250812...,150,408,NaN,NaN,FEYLKLLGKGTFGKVILVKEKATGRYYAMKILKKEVIVAKDEVAHT...,#NUM!
2,AKT2,AKT2_HUMAN,P31751,reviewed,AKT2_HUMAN,RAC-beta serine/threonine-protein kinase (EC 2...,AKT2,Homo sapiens (Human),481,"DOMAIN 5..108; /note=""PH""; /evidence=""ECO:0000...",MNEVSVIKEGWLHKRGEYIKTWRPRYFLLKSDGSFIGYKERPEAPD...,1801921; 1409633; 14702039; 15057824; 15489334...,152,409,NaN,NaN,FDYLKLLGKGTFGKVILVREKATGRYYAMKILRKEVIIAKDEVAHT...,#NUM!
3,AKT3,AKT3_HUMAN,Q9Y243,reviewed,AKT3_HUMAN,RAC-gamma serine/threonine-protein kinase (EC ...,AKT3 PKBG,Homo sapiens (Human),479,"DOMAIN 5..107; /note=""PH""; /evidence=""ECO:0000...",MSDVTIVKEGWVQKRGEYIKNWRPRYFLLKTDGSFIGYKEKPQDVD...,10092583; 10208883; 10491192; 11230166; 113873...,148,405,NaN,NaN,FDYLKLLGKGTFGKVILVREKASGKYYAMKILKKEVIIAKDEVAHT...,#NUM!
4,AURKA,AURKA_HUMAN,O14965,reviewed,AURKA_HUMAN,Aurora kinase A (EC 2.7.11.1) (Aurora 2) (Auro...,AURKA AIK AIRK1 ARK1 AURA AYK1 BTAK IAK1 STK15...,Homo sapiens (Human),403,"DOMAIN 133..383; /note=""Protein kinase""; /evid...",MDRSKENCISGPVKATAPVGGPKRVLVTQQFPCQNPLPVNSGQAQR...,9153231; 9514916; 9771714; 11780052; 15489334;...,133,383,NaN,NaN,FEIGRPLGKGKFGNVYLAREKQSKFILALKVLFKAQLEKAGVEHQL...,#NUM!


In [12]:
# For JAK3 and RPS6KA5, use the KD2 sequence as the mutations in our database fall on the second KD on those two kinases.

for p in ["JAK3", "RPS6KA5"]:
    kinase_sequences.loc[kinase_sequences[kinase_sequences["Protein"] == p].index.item(), "KD start"] = \
        int(kinase_sequences[kinase_sequences["Protein"] == p]["KD2 start"].item())

    kinase_sequences.loc[kinase_sequences[kinase_sequences["Protein"] == p].index.item(), "KD end"] = \
        int(kinase_sequences[kinase_sequences["Protein"] == p]["KD2 end"].item())


### Load Info About Mutant Kinases

In [151]:
mutant_df = pd.read_csv("mutant_df.csv")

In [152]:
mutant_df.head()

,ID,Protein,Mutation,Experimental,Fidelity,Sequence,mutant_seq,KD start,KD end,KD,in_KD,kd_mutation_loc,mutation_loc,WT_AA,mutant_KD
0,ALK_F1174L,ALK,F1174L,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,59,1174,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
1,ALK_F1245V,ALK,F1245V,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,130,1245,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
2,ALK_F1245C,ALK,F1245C,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,130,1245,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
3,ALK_I1170N,ALK,I1170N,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,55,1170,I,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
4,ALK_I1170S,ALK,I1170S,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,55,1170,I,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...


### Data Validation and Processing

**mutant_df.csv already contains the preprocessed data so the Data Validation, Generating Mutated Sequences, and Extracting Kinase Domains sections can/should be skipped.**

In [122]:
# Check if there are kinases with mutations that don"t have wildtype data
proteins_with_mutations = set(mutant_df["Protein"].unique())
kinase_sequences_of_interest = kinase_sequences[kinase_sequences["Protein"].isin(proteins_with_mutations)]
proteins_with_mutations.difference(set(kinase_sequences_of_interest.Protein.unique()))

set()

In [123]:
wt_sequence_dict = {k : v for (k,v) in zip(kinase_sequences_of_interest["Protein"], kinase_sequences_of_interest["Sequence"])}

In [124]:
# Check if wild type amino acid in "Mutation" column matches to wild type AA in sequence
mutant_df["mutation_loc"] = mutant_df["Mutation"].apply(lambda x : x[1:-1])
mutant_df["WT_AA"] = mutant_df["Mutation"].apply(lambda x : x[0])
mutant_df = mutant_df.merge(kinase_sequences[["Protein", "Sequence"]], how = "left", on = "Protein")

In [126]:
# Check if wild type amino acid in "Mutation" column matches to wild type AA in sequence
for i, row in mutant_df.iterrows():
    if row["Sequence"][int(row["mutation_loc"]) - 1] != row["WT_AA"]:
        print(row["ID"], "does not match")

### Generate Mutated Sequences

In [127]:
def make_mutation(idx):
    try:
        protein, mutation = idx.split("_")
        
        position = int(mutation[1:-1]) - 1
        mutant_aa = mutation[-1]
    
        if protein in wt_sequence_dict.keys():
            wt = wt_sequence_dict[protein]

            mutant = wt[:position] + mutant_aa + wt[position + 1:]

            return mutant

        else:
            return None
        
    except:
        print(idx)
        
        return None


In [128]:
mutant_df["mutant_seq"] = mutant_df["ID"].apply(lambda x : make_mutation(x))

In [129]:
mutant_df[mutant_df["mutant_seq"].isna()]

,ID,Protein,Mutation,Experimental,Fidelity,mutant_seq,KD start,KD end,KD,in_KD,kd_mutation_loc,mutation_loc,WT_AA,Sequence


In [130]:
mutant_df[mutant_df.duplicated()]

,ID,Protein,Mutation,Experimental,Fidelity,mutant_seq,KD start,KD end,KD,in_KD,kd_mutation_loc,mutation_loc,WT_AA,Sequence


In [131]:
mutant_df[mutant_df["ID"].str.contains("\*")]

,ID,Protein,Mutation,Experimental,Fidelity,mutant_seq,KD start,KD end,KD,in_KD,kd_mutation_loc,mutation_loc,WT_AA,Sequence


In [132]:
len(mutant_df)

1003

### Extracting Kinase Domains

In [134]:
def extract_kd(df, not_in_KD_mutations, original_seq_col_name):
    mut_loc = int(df["Mutation"][1:-1])
    start = df["KD start"]
    end = df["KD end"]
    
    if mut_loc < start -1 or mut_loc >= end :
        print(df["ID"], "mutation not in KD")
        not_in_KD_mutations.add(df["ID"])

    return df[original_seq_col_name][start - 1: end]
    

In [135]:
not_in_KD_mutations = set()
mutant_df["KD"] = mutant_df.apply(extract_kd, axis = 1, args = (not_in_KD_mutations, "Sequence", ))

ALK_F1098V mutation not in KD
ALK_Y1096A mutation not in KD
ALK_C1097A mutation not in KD
BRAF_W450L mutation not in KD
EGFR_E709A mutation not in KD
EGFR_E709K mutation not in KD
EGFR_K708M mutation not in KD
EGFR_E709G mutation not in KD
EGFR_E709Q mutation not in KD
EGFR_G983R mutation not in KD
EGFR_A702S mutation not in KD
EGFR_E690K mutation not in KD
EGFR_E697E mutation not in KD
EGFR_P699L mutation not in KD
EGFR_P699S mutation not in KD
ABL1_Y226C mutation not in KD
AKT1_Y474F mutation not in KD


In [140]:
mutant_df["mutant_KD"] = mutant_df.apply(extract_kd, axis = 1, args = (not_in_KD_mutations, "mutant_seq", ))

ALK_F1098V mutation not in KD
ALK_Y1096A mutation not in KD
ALK_C1097A mutation not in KD
BRAF_W450L mutation not in KD
EGFR_E709A mutation not in KD
EGFR_E709K mutation not in KD
EGFR_K708M mutation not in KD
EGFR_E709G mutation not in KD
EGFR_E709Q mutation not in KD
EGFR_G983R mutation not in KD
EGFR_A702S mutation not in KD
EGFR_E690K mutation not in KD
EGFR_E697E mutation not in KD
EGFR_P699L mutation not in KD
EGFR_P699S mutation not in KD
ABL1_Y226C mutation not in KD
AKT1_Y474F mutation not in KD


In [141]:
# Annotate whether mutation is in KD or not (some are outside)
mutant_df["in_KD"] = mutant_df["ID"].apply(lambda x : 0 if x in not_in_KD_mutations else 1)

In [142]:
# Translate mutation location into KD location
mutant_df["kd_mutation_loc"] = mutant_df["Mutation"].str.slice(1,-1).astype(int) - mutant_df["KD start"].astype(int) + 1

In [149]:
mutant_df.head()

,ID,Protein,Mutation,Experimental,Fidelity,Sequence,mutant_seq,KD start,KD end,KD,in_KD,kd_mutation_loc,mutation_loc,WT_AA,mutant_KD
0,ALK_F1174L,ALK,F1174L,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,59,1174,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
1,ALK_F1245V,ALK,F1245V,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,130,1245,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
2,ALK_F1245C,ALK,F1245C,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,130,1245,F,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
3,ALK_I1170N,ALK,I1170N,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,55,1170,I,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...
4,ALK_I1170S,ALK,I1170S,1,1,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,MGAIGLLWLLPLLLSTAAVGSGMGTGQRAGSPAAGPPLQPREPLSY...,1116,1392,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...,1,55,1170,I,ITLIRGLGHGAFGEVYEGQVSGMPNDPSPLQVAVKTLPEVCSEQDE...


### Generate ESM2 Embeddings

In [42]:

device = torch.device("cuda")
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()


model.to(device)
batch_converter = alphabet.get_batch_converter()

model.eval()


ESM2(
  (embed_tokens): Embedding(33, 1280, padding_idx=1)
  (layers): ModuleList(
    (0-32): 33 x TransformerLayer(
      (self_attn): MultiheadAttention(
        (k_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (v_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (q_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (out_proj): Linear(in_features=1280, out_features=1280, bias=True)
        (rot_emb): RotaryEmbedding()
      )
      (self_attn_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
      (fc1): Linear(in_features=1280, out_features=5120, bias=True)
      (fc2): Linear(in_features=5120, out_features=1280, bias=True)
      (final_layer_norm): LayerNorm((1280,), eps=1e-05, elementwise_affine=True)
    )
  )
  (contact_head): ContactPredictionHead(
    (regression): Linear(in_features=660, out_features=1, bias=True)
    (activation): Sigmoid()
  )
  (emb_layer_norm_after): LayerNorm((1280,), eps=1

In [47]:
wt_seq = [(tuple[0] + "_wt", tuple[1]) for tuple in set(zip(kinase_sequences["Protein"], kinase_sequences["Sequence"]))]

In [48]:
mutant_seq = list(zip(mutant_df["ID"].tolist(), mutant_df["mutant_seq"].tolist()))

In [52]:
def generate_embeddings(sequences):
    start = time.time()
    sequence_representations = []

    for index in tqdm(range(0, len(sequences))):
        try:
            data = [sequences[index]]
            
            batch_labels, batch_strs, batch_tokens = batch_converter(data)

            batch_tokens = batch_tokens.to(device)
    
            with torch.no_grad():
                results = model(batch_tokens, repr_layers=[33], return_contacts=False)
        
        except:
            print("Could not load", batch_labels)
            
            del batch_tokens
            torch.cuda.empty_cache()
            continue
            
    
        results = {
            key: {nested_key: value.detach().to("cpu") for nested_key, value in nested_dict.items()}
            if isinstance(nested_dict, dict)
            else nested_dict.detach().to("cpu")
            for key, nested_dict in results.items()
        }
        
        token_representations = results["representations"][33].detach().cpu()
        batch_tokens = batch_tokens.detach().to("cpu")
        
    
        sequence_representations.append((sequences[index][0], token_representations[0]))
            
        torch.cuda.empty_cache()
    
    end = time.time()
    print(end - start)


    return sequence_representations

In [53]:
wt_seq_representations = generate_embeddings(wt_seq)

  0%|          | 0/196 [00:00<?, ?it/s]

100%|██████████| 196/196 [00:40<00:00,  4.86it/s]

40.394593954086304


In [64]:
# wt_seq_representations = [(name, embedding.numpy()) for name, embedding in wt_seq_representations]

# with open("whole_protein_wt_embeddings.pkl", "wb") as file:
#   pickle.dump(wt_seq_representations, file)

In [56]:
mutant_seq_representations = generate_embeddings(mutant_seq)

100%|██████████| 1003/1003 [03:43<00:00,  4.48it/s]

223.83440685272217


In [67]:
# mutant_seq_representations = [(name, embedding.numpy()) for name, embedding in mutant_seq_representations]

# with open("whole_protein_mutant_embeddings.pkl", "wb") as file:
#   pickle.dump(mutant_seq_representations, file)

### Whole Protein First Token

In [68]:
with open("whole_protein_wt_embeddings.pkl", "rb") as file:
    ALL_wt_data = pickle.load(file)

ALL_wt_dict = dict(ALL_wt_data)
ALL_wt_embeddings = [protein_embeddings[0] for protein_embeddings in list(ALL_wt_dict.values())]
print(len(ALL_wt_embeddings))
whole_protein_first_token_scaler = StandardScaler()
whole_protein_wt_first_token_embeddings_standardized = whole_protein_first_token_scaler.fit_transform(ALL_wt_embeddings)

whole_protein_first_token_pca = PCA(n_components=100)
whole_protein_first_token_pca.fit(whole_protein_wt_first_token_embeddings_standardized)

196


PCA(n_components=100)

In [69]:
with open("whole_protein_mutant_embeddings.pkl", "rb") as file:
    mutant_seq_representations = pickle.load(file)


In [71]:
mutant_whole_protein_first_token_list = [(name, embedding[0]) for name, embedding in mutant_seq_representations]
df = pd.DataFrame(mutant_whole_protein_first_token_list , columns = ["ID", "mutant_first_token_embedding"])

df["wt_first_token_embedding"] = df["ID"].apply(lambda x : ALL_wt_dict[x.split("_")[0] + "_wt"][0])


In [72]:
# Create PCA representations
for embedding in ["mutant_first_token_embedding", "wt_first_token_embedding"]:
    df[f"{'_'.join(embedding.split('_')[:-1])}_pca"] = df[embedding].apply(lambda x : whole_protein_first_token_pca.transform(whole_protein_first_token_scaler.transform(x.reshape(1, -1))).squeeze())

In [77]:
df = df.merge(mutant_df[["ID", "in_KD"]], on="ID")

In [79]:
mutant_ESM_df = df["mutant_first_token_embedding"].apply(pd.Series).add_prefix("mutant_first_token_ESM_")
wt_ESM_df = df["wt_first_token_embedding"].apply(pd.Series).add_prefix("wt_first_token_ESM_")
mutant_PCA_df = df["mutant_first_token_pca"].apply(pd.Series).add_prefix("mutant_first_token_PCA_")
wt_PCA_df = df["wt_first_token_pca"].apply(pd.Series).add_prefix("wt_first_token_PCA_")

df = pd.concat([df, mutant_ESM_df, wt_ESM_df, mutant_PCA_df, wt_PCA_df], axis = 1)
df= df.drop(columns = ["mutant_first_token_embedding", "wt_first_token_embedding", "mutant_first_token_pca", "wt_first_token_pca"])

In [80]:
df.head()

,ID,in_KD,mutant_first_token_ESM_0,mutant_first_token_ESM_1,mutant_first_token_ESM_2,mutant_first_token_ESM_3,mutant_first_token_ESM_4,mutant_first_token_ESM_5,mutant_first_token_ESM_6,mutant_first_token_ESM_7,...,wt_first_token_PCA_90,wt_first_token_PCA_91,wt_first_token_PCA_92,wt_first_token_PCA_93,wt_first_token_PCA_94,wt_first_token_PCA_95,wt_first_token_PCA_96,wt_first_token_PCA_97,wt_first_token_PCA_98,wt_first_token_PCA_99
0,ALK_F1174L,1,0.046016,-0.016255,0.102002,0.045932,0.020795,0.010865,0.032878,-0.039793,...,-2.329956,-0.720194,-0.453576,-2.115375,-1.76194,0.711656,2.194398,-0.278223,-1.204536,0.100896
1,ALK_F1245V,1,0.045870,-0.016686,0.102553,0.046056,0.020512,0.010943,0.033214,-0.039741,...,-2.329956,-0.720194,-0.453576,-2.115375,-1.76194,0.711656,2.194398,-0.278223,-1.204536,0.100896
2,ALK_F1245C,1,0.046175,-0.016312,0.102426,0.045879,0.021209,0.011197,0.033611,-0.039688,...,-2.329956,-0.720194,-0.453576,-2.115375,-1.76194,0.711656,2.194398,-0.278223,-1.204536,0.100896
3,ALK_I1170N,1,0.046299,-0.016886,0.101906,0.046237,0.020496,0.011243,0.033468,-0.039943,...,-2.329956,-0.720194,-0.453576,-2.115375,-1.76194,0.711656,2.194398,-0.278223,-1.204536,0.100896
4,ALK_I1170S,1,0.046249,-0.017117,0.102262,0.046173,0.020509,0.011036,0.033413,-0.039728,...,-2.329956,-0.720194,-0.453576,-2.115375,-1.76194,0.711656,2.194398,-0.278223,-1.204536,0.100896


In [81]:
# df.to_csv("whole_protein_first_token_esm_and_pca_embeddings.csv")

### Whole Protein Missense Token

In [92]:
with open("whole_protein_wt_embeddings.pkl", "rb") as file:
    ALL_wt_data = pickle.load(file)

ALL_wt_dict = dict(ALL_wt_data)
ALL_wt_dict = {k:v[1:-1] for (k,v) in ALL_wt_dict.items()}

ALL_wt_embeddings = np.stack([embedding for protein_embeddings in ALL_wt_dict.values() for embedding in protein_embeddings])
print(len(ALL_wt_embeddings))
whole_protein_missense_token_scaler = StandardScaler()
whole_protein_wt_all_amino_acids_embeddings_standardized = whole_protein_missense_token_scaler.fit_transform(ALL_wt_embeddings)

whole_protein_missense_token_pca = PCA(n_components=100)
whole_protein_missense_token_pca.fit(whole_protein_wt_all_amino_acids_embeddings_standardized)

149718


PCA(n_components=100)

In [155]:
mut_loc_dict = {k:int(v) for k,v in zip(mutant_df["ID"], mutant_df["mutation_loc"])}

In [163]:
with open("whole_protein_mutant_embeddings.pkl", "rb") as file:
    mutant_seq_representations = pickle.load(file)

In [164]:
# Remove first token which is CLS token from the protein langauge model
mutant_all_token_dict = {k: v[1:-1] for k,v in mutant_seq_representations }

In [165]:
df_mutated_AA = pd.DataFrame(mutant_df[["ID", "in_KD"]])


df_mutated_AA["wt_mutated_AA_token"] = df_mutated_AA["ID"].apply(lambda x : ALL_wt_dict[x.split("_")[0] + "_wt"][mut_loc_dict[x]-1]) # First token has been removed to train PCA on residue embeddings only so we adjust index accordingly
df_mutated_AA["mutant_AA_token"] = df_mutated_AA["ID"].apply(lambda x : mutant_all_token_dict[x][mut_loc_dict[x]])



In [168]:
for embedding in ["wt_mutated_AA_token", "mutant_AA_token"]:

    df_mutated_AA[f"{'_'.join(embedding.split('_')[:-1])}_pca"] = df_mutated_AA[embedding].apply(lambda x : whole_protein_missense_token_pca.transform(whole_protein_missense_token_scaler.transform(x.reshape(1, -1))).squeeze())

In [169]:
mutant_ESM_df = df_mutated_AA["mutant_AA_token"].apply(pd.Series).add_prefix("mutant_ESM_")
wt_ESM_df = df_mutated_AA["wt_mutated_AA_token"].apply(pd.Series).add_prefix("wt_ESM_")
mutant_PCA_df = df_mutated_AA["mutant_AA_pca"].apply(pd.Series).add_prefix("mutant_PCA_")
wt_PCA_df = df_mutated_AA["wt_mutated_AA_pca"].apply(pd.Series).add_prefix("wt_PCA_")

df_mutated_AA = pd.concat([df_mutated_AA, mutant_ESM_df, wt_ESM_df, mutant_PCA_df, wt_PCA_df], axis = 1)
df_mutated_AA = df_mutated_AA.drop(columns = ["wt_mutated_AA_token", "mutant_AA_token", "wt_mutated_AA_pca", "mutant_AA_pca"])

In [170]:
df_mutated_AA.head()

,ID,in_KD,mutant_ESM_0,mutant_ESM_1,mutant_ESM_2,mutant_ESM_3,mutant_ESM_4,mutant_ESM_5,mutant_ESM_6,mutant_ESM_7,...,wt_PCA_90,wt_PCA_91,wt_PCA_92,wt_PCA_93,wt_PCA_94,wt_PCA_95,wt_PCA_96,wt_PCA_97,wt_PCA_98,wt_PCA_99
0,ALK_F1174L,1,0.460479,0.048090,0.231594,0.021890,0.018112,0.338462,0.096668,-0.087058,...,0.095775,0.318608,1.604843,1.728218,-1.749925,-1.427386,-1.143305,-1.039397,-2.818923,-2.278226
1,ALK_F1245V,1,-0.222916,-0.230643,-0.185541,-0.217733,-0.131445,-0.221990,-0.003691,-0.037011,...,-1.256793,0.337100,-1.281281,-4.658317,-1.453456,-1.227803,3.752722,-0.911660,0.148250,0.673166
2,ALK_F1245C,1,-0.161711,-0.194445,-0.193848,-0.135951,-0.164812,-0.263547,-0.018845,0.030605,...,-1.256793,0.337100,-1.281281,-4.658317,-1.453456,-1.227803,3.752722,-0.911660,0.148250,0.673166
3,ALK_I1170N,1,-0.006527,-0.196725,-0.307611,0.353425,-0.272354,0.037702,0.166402,0.160167,...,-0.666713,3.240715,-2.329483,1.446853,-2.972416,1.990175,0.205153,1.370245,-0.904864,-0.264454
4,ALK_I1170S,1,-0.041783,-0.190775,-0.308923,0.358562,-0.245173,0.027169,0.170443,0.155917,...,-0.666713,3.240715,-2.329483,1.446853,-2.972416,1.990175,0.205153,1.370245,-0.904864,-0.264454


In [171]:
# df_mutated_AA.to_csv("whole_protein_mutation_token_esm_and_pca_embeddings.csv")


## Kinase Domain

In [173]:
wt_kd_seq = [(tuple[0] + "_wt", tuple[1]) for tuple in set(zip(mutant_df["Protein"], mutant_df["KD"]))]

In [174]:
mutant_kd_seq = list(zip(mutant_df["ID"].tolist(), mutant_df["mutant_KD"].tolist()))

In [175]:
wt_kd_representations = generate_embeddings(wt_kd_seq)

100%|██████████| 196/196 [00:17<00:00, 11.45it/s]

17.128703355789185


In [178]:
# wt_kd_representations = [(name, embedding.numpy()) for name, embedding in wt_kd_representations]

# with open("kinase_domain_wt_embeddings.pkl", "wb") as file:
#   pickle.dump(wt_kd_representations, file)

In [176]:
mutant_kd_representations = generate_embeddings(mutant_kd_seq)

100%|██████████| 1003/1003 [01:22<00:00, 12.16it/s]

82.49173879623413


In [187]:
#mutant_kd_representations = [(name, embedding.numpy()) for name, embedding in mutant_kd_representations]

# with open("kinase_domain_mutant_embeddings.pkl", "wb") as file:
#   pickle.dump(mutant_kd_representations, file)

### KD First Token

In [181]:
with open("kinase_domain_wt_embeddings.pkl", "rb") as file:
    ALL_wt_data = pickle.load(file)

ALL_wt_dict = dict(ALL_wt_data)
ALL_wt_embeddings = [protein_embeddings[0] for protein_embeddings in list(ALL_wt_dict.values())]
print(len(ALL_wt_embeddings))
kd_first_token_scaler = StandardScaler()
kd_wt_first_token_embeddings_standardized = kd_first_token_scaler.fit_transform(ALL_wt_embeddings)

kd_first_token_pca = PCA(n_components=100)
kd_first_token_pca.fit(kd_wt_first_token_embeddings_standardized)

196


PCA(n_components=100)

In [188]:
with open("whole_protein_mutant_embeddings.pkl", "rb") as file:
    mutant_seq_kd_representations = pickle.load(file)


In [190]:
mutant_kd_first_token_list = [(name, embedding[0]) for name, embedding in mutant_seq_kd_representations]
df = pd.DataFrame(mutant_kd_first_token_list , columns = ["ID", "mutant_first_token_embedding"])

df["wt_first_token_embedding"] = df["ID"].apply(lambda x : ALL_wt_dict[x.split("_")[0] + "_wt"][0])


In [193]:
for embedding in ["mutant_first_token_embedding", "wt_first_token_embedding"]:
    df[f"{'_'.join(embedding.split('_')[:-1])}_pca"] = df[embedding].apply(lambda x : whole_protein_first_token_pca.transform(whole_protein_first_token_scaler.transform(x.reshape(1, -1))).squeeze())

In [194]:
in_kd_dict = {k: v for k, v in zip(mutant_df["ID"], mutant_df["in_KD"])}
df["in_KD"] = df["ID"].apply(lambda x : 0 if x in not_in_KD_mutations else 1)

In [195]:
mutant_ESM_df = df["mutant_first_token_embedding"].apply(pd.Series).add_prefix("mutant_first_token_ESM_")
wt_ESM_df = df["wt_first_token_embedding"].apply(pd.Series).add_prefix("wt_first_token_ESM_")
mutant_PCA_df = df["mutant_first_token_pca"].apply(pd.Series).add_prefix("mutant_first_token_PCA_")
wt_PCA_df = df["wt_first_token_pca"].apply(pd.Series).add_prefix("wt_first_token_PCA_")

df = pd.concat([df, mutant_ESM_df, wt_ESM_df, mutant_PCA_df, wt_PCA_df], axis = 1)
df= df.drop(columns = ["mutant_first_token_embedding", "wt_first_token_embedding", "mutant_first_token_pca", "wt_first_token_pca"])

In [196]:
df.head()

,ID,in_KD,mutant_first_token_ESM_0,mutant_first_token_ESM_1,mutant_first_token_ESM_2,mutant_first_token_ESM_3,mutant_first_token_ESM_4,mutant_first_token_ESM_5,mutant_first_token_ESM_6,mutant_first_token_ESM_7,...,wt_first_token_PCA_90,wt_first_token_PCA_91,wt_first_token_PCA_92,wt_first_token_PCA_93,wt_first_token_PCA_94,wt_first_token_PCA_95,wt_first_token_PCA_96,wt_first_token_PCA_97,wt_first_token_PCA_98,wt_first_token_PCA_99
0,ALK_F1174L,1,0.046016,-0.016255,0.102002,0.045932,0.020795,0.010865,0.032878,-0.039793,...,1.768479,1.716016,4.416175,-3.304655,1.931377,-1.566953,-4.347733,-0.984782,-3.27583,-4.163978
1,ALK_F1245V,1,0.045870,-0.016686,0.102553,0.046056,0.020512,0.010943,0.033214,-0.039741,...,1.768479,1.716016,4.416175,-3.304655,1.931377,-1.566953,-4.347733,-0.984782,-3.27583,-4.163978
2,ALK_F1245C,1,0.046175,-0.016312,0.102426,0.045879,0.021209,0.011197,0.033611,-0.039688,...,1.768479,1.716016,4.416175,-3.304655,1.931377,-1.566953,-4.347733,-0.984782,-3.27583,-4.163978
3,ALK_I1170N,1,0.046299,-0.016886,0.101906,0.046237,0.020496,0.011243,0.033468,-0.039943,...,1.768479,1.716016,4.416175,-3.304655,1.931377,-1.566953,-4.347733,-0.984782,-3.27583,-4.163978
4,ALK_I1170S,1,0.046249,-0.017117,0.102262,0.046173,0.020509,0.011036,0.033413,-0.039728,...,1.768479,1.716016,4.416175,-3.304655,1.931377,-1.566953,-4.347733,-0.984782,-3.27583,-4.163978


In [197]:
# df.to_csv("kinase_domain_first_token_esm_and_pca_embeddings.csv")

### KD Missense Token

In [198]:
with open("kinase_domain_wt_embeddings.pkl", "rb") as file:
    ALL_wt_data = pickle.load(file)

ALL_wt_dict = dict(ALL_wt_data)
ALL_wt_dict = {k:v[1:-1] for (k,v) in ALL_wt_dict.items()}

ALL_wt_embeddings = np.stack([embedding for protein_embeddings in ALL_wt_dict.values() for embedding in protein_embeddings])
print(len(ALL_wt_embeddings))
kd_missense_token_scaler = StandardScaler()
kd_wt_all_amino_acids_embeddings_standardized = kd_missense_token_scaler.fit_transform(ALL_wt_embeddings)

kd_missense_token_pca = PCA(n_components=100)
kd_missense_token_pca.fit(kd_wt_all_amino_acids_embeddings_standardized)

53898


PCA(n_components=100)

In [199]:
with open("kinase_domain_mutant_embeddings.pkl", "rb") as file:
    mutant_kd_representations  = pickle.load(file)

In [200]:
mut_loc_dict = {k:int(v) for k,v in zip(mutant_df["ID"], mutant_df["kd_mutation_loc"])}
mutant_all_token_dict = {k: v[1:-1] for k,v in mutant_kd_representations}

In [201]:
# Filter out mutations outside kinase domain with mutated AA embedding out of bounds
mutant_df_in_kd = mutant_df[mutant_df["in_KD"] == 1]

In [202]:
df_mutated_AA = pd.DataFrame(mutant_df_in_kd[["ID", "in_KD"]])


df_mutated_AA["wt_mutated_AA_token"] = df_mutated_AA["ID"].apply(lambda x : ALL_wt_dict[x.split("_")[0] + "_wt"][mut_loc_dict[x]-1]) # First token has been removed to train PCA so we adjust index accordingly
df_mutated_AA["mutant_AA_token"] = df_mutated_AA["ID"].apply(lambda x : mutant_all_token_dict[x][mut_loc_dict[x]])



In [204]:
for embedding in ["wt_mutated_AA_token", "mutant_AA_token"]:
    df_mutated_AA[f"{'_'.join(embedding.split('_')[:-1])}_pca"] = df_mutated_AA[embedding].apply(lambda x : kd_missense_token_pca.transform(kd_missense_token_scaler.transform(x.reshape(1, -1))).squeeze())

In [205]:
mutant_ESM_df = df_mutated_AA["mutant_AA_token"].apply(pd.Series).add_prefix("mutant_ESM_")
wt_ESM_df = df_mutated_AA["wt_mutated_AA_token"].apply(pd.Series).add_prefix("wt_ESM_")
mutant_PCA_df = df_mutated_AA["mutant_AA_pca"].apply(pd.Series).add_prefix("mutant_PCA_")
wt_PCA_df = df_mutated_AA["wt_mutated_AA_pca"].apply(pd.Series).add_prefix("wt_PCA_")

df_mutated_AA = pd.concat([df_mutated_AA, mutant_ESM_df, wt_ESM_df, mutant_PCA_df, wt_PCA_df], axis = 1)
df_mutated_AA = df_mutated_AA.drop(columns = ["wt_mutated_AA_token", "mutant_AA_token", "wt_mutated_AA_pca", "mutant_AA_pca"])

In [206]:
df_mutated_AA.head()

,ID,in_KD,mutant_ESM_0,mutant_ESM_1,mutant_ESM_2,mutant_ESM_3,mutant_ESM_4,mutant_ESM_5,mutant_ESM_6,mutant_ESM_7,...,wt_PCA_90,wt_PCA_91,wt_PCA_92,wt_PCA_93,wt_PCA_94,wt_PCA_95,wt_PCA_96,wt_PCA_97,wt_PCA_98,wt_PCA_99
0,ALK_F1174L,1,0.388936,0.138129,0.283476,-0.195227,0.072271,0.334112,-0.099708,-0.040604,...,1.328777,-1.759166,-0.743201,0.611641,2.110884,0.694915,1.474894,2.097901,0.723519,1.294101
1,ALK_F1245V,1,-0.259482,-0.202089,-0.263377,-0.305441,-0.135409,-0.238090,-0.012242,-0.105182,...,1.771029,-1.618016,-1.824278,-0.227553,0.111385,0.086062,1.153615,2.519463,-3.144527,-0.211856
2,ALK_F1245C,1,-0.232394,-0.194110,-0.234137,-0.200489,-0.164098,-0.293934,-0.051302,0.008021,...,1.771029,-1.618016,-1.824278,-0.227553,0.111385,0.086062,1.153615,2.519463,-3.144527,-0.211856
3,ALK_I1170N,1,-0.059908,-0.278425,-0.382416,0.243875,-0.392162,-0.060528,0.188877,0.111965,...,-1.540583,2.879583,0.414483,0.975181,0.658012,0.662251,2.327931,1.167575,-0.011728,0.454219
4,ALK_I1170S,1,-0.101843,-0.232568,-0.400135,0.255190,-0.368141,-0.080996,0.158029,0.127364,...,-1.540583,2.879583,0.414483,0.975181,0.658012,0.662251,2.327931,1.167575,-0.011728,0.454219


In [207]:
# df_mutated_AA.to_csv("kinase_domain_mutation_token_esm_and_pca_embeddings.csv")